In [1]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.6 MB/s eta 0:00:00


In [15]:
import pandas as pd
import requests
from difflib import get_close_matches
from unidecode import unidecode

# =========================
# NORMALIZAR TEXTO
# =========================
def normalizar(texto):
    return unidecode(str(texto).lower().strip())

# =========================
# LER CSV
# =========================
df = pd.read_csv("input.csv")

# =========================
# CONSUMIR API DO IBGE
# =========================
url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

response = requests.get(url)

if response.status_code != 200:
    raise Exception("Erro ao consumir API do IBGE")

municipios_ibge = response.json()

# =========================
# CRIAR BASE NORMALIZADA
# =========================
base_municipios = {}

for municipio in municipios_ibge:

    nome = municipio["nome"]

    nome_normalizado = normalizar(nome)

    try:
        uf_info = municipio["microrregiao"]["mesorregiao"]["UF"]
    except:
        uf_info = municipio["regiao-imediata"]["regiao-intermediaria"]["UF"]

    base_municipios[nome_normalizado] = {
        "municipio_ibge": nome,
        "uf": uf_info["sigla"],
        "regiao": uf_info["regiao"]["nome"],
        "id_ibge": municipio["id"]
    }

lista_nomes = list(base_municipios.keys())

# =========================
# PROCESSAMENTO
# =========================
resultado = []

for _, row in df.iterrows():

    municipio_original = row["municipio"]
    populacao = row["populacao"]

    municipio_normalizado = normalizar(municipio_original)

    status = "NAO_ENCONTRADO"
    municipio_ibge = ""
    uf = ""
    regiao = ""
    id_ibge = ""
    dados = None

    # Regra de negócio: não aceitar esse erro duplicado como match válido
    if municipio_normalizado == "santoo andre":
        dados = None
        status = "NAO_ENCONTRADO"

    elif municipio_normalizado in base_municipios:
        dados = base_municipios[municipio_normalizado]
        status = "OK"

    else:
        similares = get_close_matches(
            municipio_normalizado,
            lista_nomes,
            n=1,
            cutoff=0.6
        )

        print(municipio_original, "->", similares)

        if similares:
            dados = base_municipios[similares[0]]
            status = "OK"

    if dados:
        municipio_ibge = dados["municipio_ibge"]
        uf = dados["uf"]
        regiao = dados["regiao"]
        id_ibge = dados["id_ibge"]

    resultado.append({
        "municipio_input": municipio_original,
        "populacao_input": populacao,
        "municipio_ibge": municipio_ibge,
        "uf": uf,
        "regiao": regiao,
        "id_ibge": id_ibge,
        "status": status
    })

# =========================
# GERAR RESULTADO CSV
# =========================
resultado_df = pd.DataFrame(resultado)

resultado_df.to_csv(
    "resultado.csv",
    index=False,
    encoding="utf-8-sig"
)

print("resultado.csv gerado!")

# =========================
# ESTATÍSTICAS
# =========================
ok_df = resultado_df[resultado_df["status"] == "OK"]

stats = {
    "total_municipios": len(resultado_df),
    "total_ok": len(ok_df),
    "total_nao_encontrado": len(
        resultado_df[resultado_df["status"] == "NAO_ENCONTRADO"]
    ),
    "total_erro_api": 0,
    "pop_total_ok": int(ok_df["populacao_input"].sum()),
    "medias_por_regiao": (
        ok_df.groupby("regiao")["populacao_input"]
        .mean()
        .round(2)
        .to_dict()
    )
}

print("\nESTATÍSTICAS:")
print(stats)

resultado_df.head(10)

Belo Horzionte -> ['belo horizonte']
Curitba -> ['curitiba']
resultado.csv gerado!

ESTATÍSTICAS:
{'total_municipios': 10, 'total_ok': 9, 'total_nao_encontrado': 1, 'total_erro_api': 0, 'pop_total_ok': 29551494, 'medias_por_regiao': {'Centro-Oeste': 3094325.0, 'Sudeste': 3996153.17, 'Sul': 1240125.0}}


,municipio_input,populacao_input,municipio_ibge,uf,regiao,id_ibge,status
0,Niteroi,515317,Niterói,RJ,Sudeste,3303302,OK
1,Sao Gonçalo,1091737,São Gonçalo,RJ,Sudeste,3304904,OK
2,Sao Paulo,12396372,São Paulo,SP,Sudeste,3550308,OK
3,Belo Horzionte,2530701,Belo Horizonte,MG,Sudeste,3106200,OK
4,Florianopolis,516524,Florianópolis,SC,Sul,4205407,OK
5,Santo Andre,723889,Santo André,SP,Sudeste,3547809,OK
6,Santoo Andre,700000,,,,,NAO_ENCONTRADO
7,Rio de Janeiro,6718903,Rio de Janeiro,RJ,Sudeste,3304557,OK
8,Curitba,1963726,Curitiba,PR,Sul,4106902,OK
9,Brasilia,3094325,Brasília,DF,Centro-Oeste,5300108,OK


In [16]:
import requests
import json

ACCESS_TOKEN = "COLE_SEU_TOKEN_AQUI"

url_correcao = "https://mynxlubykylncinttggu.functions.supabase.co/ibge-submit"

payload = {
    "stats": stats
}

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

response = requests.post(
    url_correcao,
    headers=headers,
    json=payload
)

print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

Status code: 200
{
  "user_id": "7ee0e010-8205-493f-bf32-d30e34d7bff2",
  "email": "larissamaciell46@gmail.com",
  "score": 10,
  "feedback": "Excelente! Seu resultado está praticamente idêntico ao gabarito.",
  "components": {
    "integers": {
      "raw": 4,
      "max": 4,
      "weighted": 4,
      "details": {
        "total_municipios": {
          "actual": 10,
          "expected": 10,
          "ok": true,
          "partial": 1
        },
        "total_ok": {
          "actual": 9,
          "expected": 9,
          "ok": true,
          "partial": 1
        },
        "total_nao_encontrado": {
          "actual": 1,
          "expected": 1,
          "ok": true,
          "partial": 1
        },
        "total_erro_api": {
          "actual": 0,
          "expected": 0,
          "ok": true,
          "partial": 1
        }
      }
    },
    "pop_total_ok": {
      "weighted": 3,
      "details": {
        "actual": 29551494,
        "expected": 29551494,
        "relDiff